In [1]:
from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.Resize((64,64)),
    transforms.ToTensor()
])

dataset = datasets.ImageFolder(
    "EuroSAT",
    transform=transform
)

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'EuroSAT'

In [2]:
import os

print(os.getcwd())

C:\Users\RAGHAV


In [3]:
import os

print(os.listdir())

['.anaconda', '.arduinoIDE', '.conda', '.condarc', '.continuum', '.cursor', '.ipynb_checkpoints', '.ipython', '.jupyter', '.matplotlib', '.spyder-py3', '.vscode', '3D Objects', 'anaconda3', 'AppData', 'Application Data', 'BullseyeCoverageError.txt', 'ComputingAssignment_CASESTUDY1.py', 'ComputingAssignment_CASESTUDY2.py', 'ComputingAssignment_CASESTUDY3.py', 'ComputingAssignment_CASESTUDY4.py', 'ComputingAssignment_CASESTUDY5.py', 'ComputingAssignment_Q1.py', 'ComputingAssignment_Q2.py', 'ComputingAssignment_Q3.py', 'ComputingAssignment_Q4.py', 'ComputingAssignment_Q5.py', 'Contacts', 'Cookies', 'Desktop', 'Documents', 'Downloads', 'eurosat_cnn.ipynb', 'Favorites', 'IIT-JEE', 'IntelGraphicsProfiles', 'Jedi', 'Links', 'Local Settings', 'LumoraQ.ipynb', 'Music', 'My Documents', 'NetHood', 'NTUSER.DAT', 'ntuser.dat.LOG1', 'ntuser.dat.LOG2', 'NTUSER.DAT{6386d136-8e49-11ef-a652-928238c2ebc9}.TM.blf', 'NTUSER.DAT{6386d136-8e49-11ef-a652-928238c2ebc9}.TMContainer00000000000000000001.regtrans-

In [4]:
dataset = datasets.ImageFolder(
    r"C:\Users\YourName\Downloads\EuroSAT_RGB",
    transform=transform
)

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'C:\\Users\\YourName\\Downloads\\EuroSAT_RGB'

In [5]:
dataset = datasets.ImageFolder(
    r"C:\Users\YOURNAME\Downloads\EuroSAT\2750",
    transform=transform
)

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'C:\\Users\\YOURNAME\\Downloads\\EuroSAT\\2750'

In [6]:
dataset = datasets.ImageFolder(
    r"C:\Users\RAGHAV\Downloads\EuroSAT\2750",
    transform=transform
)

In [7]:
dataset = datasets.ImageFolder(
    r"C:\Users\RAGHAV\Downloads\EuroSAT\2750",
    transform=transform
)

print("Classes:", dataset.classes)
print("Number of images:", len(dataset))

Classes: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
Number of images: 27000


In [8]:
from torch.utils.data import random_split, DataLoader

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(
    dataset,
    [train_size, test_size]
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

print("Train:", len(train_dataset))
print("Test:", len(test_dataset))

Train: 21600
Test: 5400


In [9]:
import torch
import torch.nn as nn

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2,2)

        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)

        self.fc1 = nn.Linear(32*16*16, 128)
        self.fc2 = nn.Linear(128, 10)

        self.relu = nn.ReLU()

    def forward(self, x):

        x = self.pool(self.relu(self.conv1(x)))

        x = self.pool(self.relu(self.conv2(x)))

        x = x.view(x.size(0), -1)

        x = self.relu(self.fc1(x))

        x = self.fc2(x)

        return x

model = SimpleCNN()

print(model)

SimpleCNN(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=8192, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
  (relu): ReLU()
)


In [10]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using:", device)

model = model.to(device)

Using: cpu


In [11]:
import torch.optim as optim
import torch.nn as nn

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

In [12]:
import torch.optim as optim
import torch.nn as nn

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

In [13]:
for epoch in range(1):

    running_loss = 0.0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    print(
        f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader):.4f}"
    )

Epoch 1, Loss: 1.2602


In [14]:
correct = 0
total = 0

model.eval()

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print(f"Accuracy: {accuracy:.2f}%")

Accuracy: 67.61%


In [15]:
for epoch in range(5):

    model.train()

    running_loss = 0.0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    print(
        f"Epoch {epoch+1}/5, Loss: {running_loss/len(train_loader):.4f}"
    )

Epoch 1/5, Loss: 0.7596
Epoch 2/5, Loss: 0.6482
Epoch 3/5, Loss: 0.5764
Epoch 4/5, Loss: 0.5174
Epoch 5/5, Loss: 0.4694


In [16]:
correct = 0
total = 0

model.eval()

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print(f"Accuracy: {accuracy:.2f}%")

Accuracy: 79.87%
